# CNN training template: real JHTDB residual super-resolution

Use this notebook to compare **CNN architectures on the same real data and split**. The data preparation, loaders, training loop, validation, checkpoint selection, and test metrics are already implemented. Members should edit only cells marked `YOUR CODE HERE`.

This notebook follows the maintained `cnn` pipeline in `src/superresolution/`:

- JHTDB provides real DNS velocity cubes from `isotropic1024coarse`;
- preprocessing Gaussian-filters, downsamples, and spline-upsamples each cube;
- on disk, `input` is that coarse-upsampled velocity field;
- on disk, `target` is the missing correction: `DNS - input`;
- the model predicts the correction and reconstruction is `input + predicted_correction`.

The maintained pipeline downloads JHTDB cubes once, caches them as `.npy` files, and creates chronological `train/`, `val/`, and `test/` directories. This notebook trains only from those cached files. It does not generate synthetic training data or call JHTDB inside a training batch.

## Before running

**New PACE users:** follow [`docs/PACE_NOTEBOOK_QUICKSTART.md`](../docs/PACE_NOTEBOOK_QUICKSTART.md). After unzipping the repository, its one required preparation command is `bash scripts/pace_prepare_notebook.sh`. Wait for the verified 490/105/105 split before clicking Run All.

From the repository root, set up the environment, then download and preprocess once using the team's configured JHTDB token:

```bash
uv sync --group extras
uv run python -m superresolution.download    # add env=hpc on PACE
uv run python -m superresolution.preprocess --config-name=cnn
uv run jupyter lab
```

The team token is already configured; do not copy it into this notebook. `JHTDB_TOKEN` is only an optional override if the shared token is rotated. On PACE, export `SUPERRES_PROJECT_DIR`, `SUPERRES_STORAGE_ROOT`, and `SUPERRES_VENV` as documented in the README, add `env=hpc` to download and preprocess, and choose the project Jupyter kernel. The default download is 700 real 128^3 cubes. Completed cubes are cached and skipped on rerun.


In [ ]:
from pathlib import Path
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Experiment settings

Give every run a unique `EXPERIMENT_NAME`; otherwise you may overwrite your own best weights. `MAX_*_SAMPLES` makes a quick plumbing test possible. Set them to `None` for the full split.

**Team comparison rule:** use the same default 700 JHTDB timesteps, preprocessing settings, and chronological 490/105/105 train/validation/test split. Separate copies in each member's PACE storage are fine because this pipeline is deterministic. Results from sample-limited runs are smoke tests and should not be compared as final experiments. For a future dataset, update `DATASET_NAME` and `EXPECTED_SPLIT_SIZES`, or set the latter to `None`.

Locally, storage defaults to the repository. On PACE, `SUPERRES_STORAGE_ROOT` selects scratch automatically. The expected layout is `train/input_tXXXX.npy`, `train/target_tXXXX.npy`, and likewise for validation and test.

In [ ]:
# Locate code and scratch storage locally or on PACE.
project_override = os.environ.get("SUPERRES_PROJECT_DIR")
REPO_ROOT = Path(project_override).expanduser() if project_override else next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "pyproject.toml").is_file()), None
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the superresolution-nn repository root.")
pace_storage = os.environ.get("SUPERRES_STORAGE_ROOT")
if pace_storage:
    STORAGE_ROOT = Path(pace_storage).expanduser()
elif REPO_ROOT.as_posix().startswith("/storage/ice1/"):
    STORAGE_ROOT = REPO_ROOT.parent / "superresolution"
else:
    STORAGE_ROOT = REPO_ROOT
# YOUR CODE HERE: if OnDemand did not inherit the variable, uncomment:
# STORAGE_ROOT = Path("/storage/ice1/.../<user>/superresolution")

# YOUR CODE HERE: choose a unique name and optionally override the processed data path.
EXPERIMENT_NAME = "your_name_cnn_v1"
DATASET_NAME = "JHTDB isotropic1024coarse"
# YOUR CODE HERE: update for a future dataset, or use None to disable this check.
EXPECTED_SPLIT_SIZES = {"train": 490, "val": 105, "test": 105}
processed_override = os.environ.get("SUPERRES_PROCESSED_DIR")
own_processed = STORAGE_ROOT / "data/processed/cnn"
PROCESSED_DIR = (Path(processed_override).expanduser()
                 if processed_override else own_processed)

# YOUR CODE HERE: shared training hyperparameters.
BATCH_SIZE = 1
NUM_WORKERS = 0       # Safe in notebooks/Windows; try 2-4 on Linux/PACE.
EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# YOUR CODE HERE: keep these as None so every member uses the complete real split.
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None
MAX_TEST_SAMPLES = None

ARTIFACT_BASE = (STORAGE_ROOT / "notebook_runs" if STORAGE_ROOT != REPO_ROOT
                 else REPO_ROOT / "superresolution_experiments/runs")
ARTIFACT_DIR = ARTIFACT_BASE / EXPERIMENT_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BEST_WEIGHTS_PATH = ARTIFACT_DIR / "best_weights.pth"
HISTORY_PATH = ARTIFACT_DIR / "history.json"
print("Data:", PROCESSED_DIR.resolve())
print("Artifacts:", ARTIFACT_DIR.resolve())


## 2. Load the fixed train/validation/test split

The repository's preprocessing performs a chronological 70/15/15 split before this notebook runs. Training data updates weights. Validation data selects the best epoch. Test data is used once for the final report.

The helper below pairs files by timestep, rather than trusting two independently sorted lists. A missing or extra file therefore causes a clear error. Arrays are stored as `(z, y, x, 3)` and converted to PyTorch's `(3, z, y, x)` channel-first format.

In [ ]:
def timestep_from_path(path):
    return path.stem.split("_t", 1)[1]


def paired_files(split_dir, limit=None):
    inputs = {timestep_from_path(p): p for p in split_dir.glob("input_t*.npy")}
    targets = {timestep_from_path(p): p for p in split_dir.glob("target_t*.npy")}
    missing_targets = sorted(set(inputs) - set(targets))
    missing_inputs = sorted(set(targets) - set(inputs))
    if missing_targets or missing_inputs:
        raise ValueError(
            f"Mismatched files in {split_dir}: "
            f"missing targets={missing_targets}, missing inputs={missing_inputs}"
        )
    keys = sorted(inputs)
    if not keys:
        raise FileNotFoundError(
            f"No real JHTDB input/target pairs in {split_dir}. "
            "Download and preprocess JHTDB into your own storage, then fix PROCESSED_DIR."
        )
    if limit is not None:
        keys = keys[:limit]
    return [(inputs[key], targets[key]) for key in keys]


class ResidualCNNDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        input_path, target_path = self.pairs[index]
        model_input = np.load(input_path).astype(np.float32)
        correction = np.load(target_path).astype(np.float32)
        if model_input.shape != correction.shape or model_input.shape[-1] != 3:
            raise ValueError(
                f"Expected matching (z, y, x, 3) arrays, got "
                f"{model_input.shape} and {correction.shape}"
            )
        model_input = np.moveaxis(model_input, -1, 0)
        correction = np.moveaxis(correction, -1, 0)
        return torch.from_numpy(model_input), torch.from_numpy(correction)


limits = {"train": MAX_TRAIN_SAMPLES, "val": MAX_VAL_SAMPLES, "test": MAX_TEST_SAMPLES}
datasets = {}
for split in ("train", "val", "test"):
    pairs = paired_files(PROCESSED_DIR / split, limits[split])
    datasets[split] = ResidualCNNDataset(pairs)
    print(f"{split:>5}: {len(pairs)} pairs; first timestep={timestep_from_path(pairs[0][0])}")
print("Dataset:", DATASET_NAME)

if EXPECTED_SPLIT_SIZES is not None and all(limit is None for limit in limits.values()):
    actual_split_sizes = {name: len(dataset) for name, dataset in datasets.items()}
    if actual_split_sizes != EXPECTED_SPLIT_SIZES:
        raise ValueError(
            f"This experiment expects split sizes {EXPECTED_SPLIT_SIZES}, "
            f"but found {actual_split_sizes}. Check the agreed dataset and preprocessing split."
        )
    print("Full team split verified:", actual_split_sizes)
elif any(limit is not None for limit in limits.values()):
    print("Sample-limited smoke test; do not compare as a final experiment.")
else:
    print("Loaded full dataset without a fixed split-size requirement.")

generator = torch.Generator().manual_seed(SEED)
loaders = {
    "train": DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                        generator=generator),
    "val": DataLoader(datasets["val"], batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda"),
    "test": DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=device.type == "cuda"),
}

sample_input, sample_correction = datasets["train"][0]
print("One input shape:", tuple(sample_input.shape))
print("One correction shape:", tuple(sample_correction.shape))
print("One DNS target shape:", tuple((sample_input + sample_correction).shape))

## 3. Write your CNN architecture

This is the main member-editable section. Your model must accept and return the same shape: `(batch, 3, z, y, x)`. It predicts a **correction**, not the final DNS field.

The starter is intentionally small. Replace its layers or write helper blocks above it. Keep the class name `YourCNN` so the rest of the notebook works. For this periodic turbulence dataset, circular padding is a reasonable starting assumption.

In [ ]:
class YourCNN(nn.Module):
    def __init__(self, hidden_channels=16):
        super().__init__()

        # ==================== YOUR CODE HERE ====================
        # Define your layers. Preserve all three spatial dimensions.
        self.network = nn.Sequential(
            nn.Conv3d(3, hidden_channels, kernel_size=3, padding=1, padding_mode="circular"),
            nn.ReLU(),
            nn.Conv3d(hidden_channels, hidden_channels, kernel_size=3, padding=1, padding_mode="circular"),
            nn.ReLU(),
            nn.Conv3d(hidden_channels, 3, kernel_size=3, padding=1, padding_mode="circular"),
        )
        # ========================================================

    def forward(self, x):
        # ==================== YOUR CODE HERE ====================
        correction = self.network(x)
        # ========================================================
        return correction


# YOUR CODE HERE: architecture settings may be changed and recorded.
model = YourCNN(hidden_channels=16).to(device)
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Trainable parameters: {trainable_parameters:,}")

## 4. Required shape and gradient checks

Run these checks after every architecture change. The small synthetic cube catches shape mistakes without loading a full 128^3 batch. The second check confirms that training can compute gradients.

In [ ]:
model.eval()  # Avoid changing BatchNorm running statistics during this check.
shape_test = torch.zeros(1, 3, 8, 8, 8, device=device)
shape_output = model(shape_test)
assert shape_output.shape == shape_test.shape, (shape_output.shape, shape_test.shape)
shape_output.square().mean().backward()
assert any(p.grad is not None for p in model.parameters() if p.requires_grad)
model.zero_grad(set_to_none=True)
print("Shape and gradient checks passed:", tuple(shape_output.shape))

## 5. Train and select the best validation epoch

The loss compares the predicted correction with the true correction, matching `src/superresolution/train.py`. The training loop updates parameters only from the training split. After every epoch, validation loss is measured without gradients. The weights from the lowest validation loss are saved.

If this cell runs out of memory, reduce `BATCH_SIZE` first. Changing the model width or sample count affects other aspects of the experiment.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)


def run_epoch(model, loader, training):
    model.train(training)
    total_squared_error = 0.0
    total_elements = 0
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for model_input, true_correction in loader:
            model_input = model_input.to(device, non_blocking=True)
            true_correction = true_correction.to(device, non_blocking=True)
            predicted_correction = model(model_input)
            loss = criterion(predicted_correction, true_correction)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_squared_error += loss.item() * true_correction.numel()
            total_elements += true_correction.numel()
    return total_squared_error / total_elements


history = {"train_mse": [], "val_mse": [], "learning_rate": []}
best_val_mse = float("inf")
best_epoch = None
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_mse = run_epoch(model, loaders["train"], training=True)
    val_mse = run_epoch(model, loaders["val"], training=False)
    learning_rate = optimizer.param_groups[0]["lr"]
    history["train_mse"].append(train_mse)
    history["val_mse"].append(val_mse)
    history["learning_rate"].append(learning_rate)

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_WEIGHTS_PATH)

    print(
        f"epoch {epoch:03d}/{EPOCHS}  train={train_mse:.4e}  "
        f"val={val_mse:.4e}  lr={learning_rate:.2e}"
    )
    scheduler.step()

history["best_epoch"] = best_epoch
history["best_val_mse"] = best_val_mse
history["elapsed_seconds"] = time.perf_counter() - start
HISTORY_PATH.write_text(json.dumps(history, indent=2))
print(f"Best epoch: {best_epoch}; val MSE: {best_val_mse:.4e}")
print(f"Saved: {BEST_WEIGHTS_PATH}")

In [ ]:
epochs = np.arange(1, len(history["train_mse"]) + 1)
plt.figure(figsize=(7, 4))
plt.semilogy(epochs, history["train_mse"], label="train")
plt.semilogy(epochs, history["val_mse"], label="validation")
plt.axvline(best_epoch, color="black", linestyle="--", alpha=0.5, label="best epoch")
plt.xlabel("Epoch")
plt.ylabel("Correction MSE")
plt.title(EXPERIMENT_NAME)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 6. Final evaluation on held-out test data

Reload the best validation weights before testing. We report errors on the reconstructed velocity field:

- baseline reconstruction = model input (no learned correction);
- model reconstruction = input + predicted correction;
- DNS truth = input + true correction.

MSE and MAE are accumulated over every velocity value, so the final result is independent of batch size. A useful model should beat the baseline on unseen data.

In [ ]:
model.load_state_dict(torch.load(BEST_WEIGHTS_PATH, map_location=device, weights_only=True))
model.eval()
totals = {"baseline_se": 0.0, "model_se": 0.0, "baseline_ae": 0.0, "model_ae": 0.0}
element_count = 0
example = None

with torch.inference_mode():
    for model_input, true_correction in loaders["test"]:
        model_input = model_input.to(device, non_blocking=True)
        true_correction = true_correction.to(device, non_blocking=True)
        predicted_correction = model(model_input)
        dns_truth = model_input + true_correction
        reconstruction = model_input + predicted_correction
        baseline_error = model_input - dns_truth
        model_error = reconstruction - dns_truth
        totals["baseline_se"] += baseline_error.square().sum().item()
        totals["model_se"] += model_error.square().sum().item()
        totals["baseline_ae"] += baseline_error.abs().sum().item()
        totals["model_ae"] += model_error.abs().sum().item()
        element_count += dns_truth.numel()
        if example is None:
            example = tuple(x[0].detach().cpu() for x in (model_input, reconstruction, dns_truth))

metrics = {
    "baseline_mse": totals["baseline_se"] / element_count,
    "model_mse": totals["model_se"] / element_count,
    "baseline_mae": totals["baseline_ae"] / element_count,
    "model_mae": totals["model_ae"] / element_count,
}
metrics["mse_improvement_percent"] = 100 * (1 - metrics["model_mse"] / metrics["baseline_mse"])
print(json.dumps(metrics, indent=2))
(ARTIFACT_DIR / "test_metrics.json").write_text(json.dumps(metrics, indent=2))

In [ ]:
model_input, reconstruction, dns_truth = example
component = 0  # 0=u, 1=v, 2=w
z_index = dns_truth.shape[1] // 2
slices = [
    model_input[component, z_index].numpy(),
    reconstruction[component, z_index].numpy(),
    dns_truth[component, z_index].numpy(),
    (reconstruction - dns_truth)[component, z_index].numpy(),
]
titles = ["Coarse upsampled baseline", "CNN reconstruction", "DNS truth", "CNN error"]
velocity_limit = max(np.abs(slices[0]).max(), np.abs(slices[1]).max(), np.abs(slices[2]).max())
error_limit = np.abs(slices[3]).max()
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for index, (ax, image, title) in enumerate(zip(axes, slices, titles)):
    limit = error_limit if index == 3 else velocity_limit
    shown = ax.imshow(image, cmap="coolwarm", vmin=-limit, vmax=limit, origin="lower")
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(shown, ax=ax, shrink=0.75)
fig.suptitle(f"Middle z-slice, velocity component {component}")
plt.tight_layout()
plt.show()

## 7. Record the experiment

Fill this in before sharing a result. Architecture comparisons are meaningful only when the data, preprocessing, sample limits, epochs, optimizer settings, and metric definitions are the same.

In [ ]:
# YOUR CODE HERE: describe what you changed and what you learned.
experiment_notes = {
    "experiment_name": EXPERIMENT_NAME,
    "dataset": DATASET_NAME,
    "author": "YOUR NAME HERE",
    "architecture_change": "DESCRIBE YOUR ARCHITECTURE HERE",
    "hypothesis": "WHAT DID YOU EXPECT TO IMPROVE?",
    "processed_data_dir": str(PROCESSED_DIR),
    "sample_limits": limits,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED,
    "trainable_parameters": trainable_parameters,
    "best_epoch": best_epoch,
    "best_val_mse": best_val_mse,
    **metrics,
}
notes_path = ARTIFACT_DIR / "experiment.json"
notes_path.write_text(json.dumps(experiment_notes, indent=2))
print(json.dumps(experiment_notes, indent=2))
print("Saved:", notes_path)

## CNN versus GNN templates

Keep this notebook CNN-only. CNN, upsampling CNN, and closure CNN can reuse much of this dense-tensor training structure, although their target meanings and spatial shapes differ. A GNN needs a separate template because it converts each grid point into a node, constructs `edge_index`, uses `torch_geometric.data.Data`, batches graphs with PyG's `DataLoader`, and reshapes node predictions for visualization. Sharing one notebook would hide those important differences behind conditionals.

Once the team has completed one fair CNN comparison, make a GNN notebook with the same experiment-report fields and test metrics so results remain comparable.